# 01 DataCheck


## 0. Research Question

**Research question:** Do students' perceptions of their weight reflect their actual BMI percentile?

**Main statistical method:** One-way ANOVA.

**Main comparison:** Mean `BMIPCT` across five `PerceptionOfWeight` groups.


## 1. Data


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
RAW_PATH = ROOT / "data" / "raw" / "YRBS_2007.csv"

TAB_DIR = ROOT / "outputs" / "tables"
TAB_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", "{:.4f}".format)

# ------------------------------------------------------------
# Load raw data
# ------------------------------------------------------------
raw = pd.read_csv(RAW_PATH)

required_cols = ["PerceptionOfWeight", "BMIPCT"]
missing_cols = [col for col in required_cols if col not in raw.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

print("Rows, columns:", raw.shape)

Rows, columns: (14041, 103)


## 2. Variable Definitions


### 2.1 Group variable: PerceptionOfWeight

- **Variable name:** `PerceptionOfWeight`
- **Role in this project:** independent / explanatory variable
- **What the variable measures:** how students perceive their own weight
- **Valid codes used:**
  - `1` = Very underweight
  - `2` = Slightly underweight
  - `3` = About the right weight
  - `4` = Slightly overweight
  - `5` = Very overweight
- **How missing or invalid values are handled:** missing or invalid values are excluded from the analysis.


In [2]:
perception_raw = raw["PerceptionOfWeight"]
valid_perception_codes = [1, 2, 3, 4, 5]

perception_check = pd.DataFrame({
    "metric": [
        "Total rows",
        "Missing values",
        "Non-missing values",
        "Code 1 count",
        "Code 2 count",
        "Code 3 count",
        "Code 4 count",
        "Code 5 count",
        "Invalid non-missing values",
        "Final valid group-variable rows",
    ],
    "value": [
        len(raw),
        int(perception_raw.isna().sum()),
        int(perception_raw.notna().sum()),
        int(perception_raw.eq(1).sum()),
        int(perception_raw.eq(2).sum()),
        int(perception_raw.eq(3).sum()),
        int(perception_raw.eq(4).sum()),
        int(perception_raw.eq(5).sum()),
        int((perception_raw.notna() & ~perception_raw.isin(valid_perception_codes)).sum()),
        int(perception_raw.isin(valid_perception_codes).sum()),
    ],
})

perception_check.to_csv(TAB_DIR / "01_perception_of_weight_data_check.csv", index=False)
display(perception_check)

,metric,value
0,Total rows,14041
1,Missing values,247
2,Non-missing values,13794
3,Code 1 count,272
4,Code 2 count,1599
5,Code 3 count,7769
6,Code 4 count,3542
7,Code 5 count,612
8,Invalid non-missing values,0
9,Final valid group-variable rows,13794


### 2.2 Response variable: BMIPCT

- **Variable name:** `BMIPCT`
- **Role in this project:** dependent / response variable
- **What the variable measures:** student BMI percentile
- **Valid values used:** numeric values from 0 to 100
- **How missing or invalid values are handled:** missing values and values outside 0 to 100 are excluded from the analysis.


In [3]:
bmipct_raw = raw["BMIPCT"]

bmipct_check = pd.DataFrame({
    "metric": [
        "Total rows",
        "Missing values",
        "Non-missing values",
        "Invalid non-missing values outside 0-100",
        "Final valid response-variable rows",
    ],
    "value": [
        len(raw),
        int(bmipct_raw.isna().sum()),
        int(bmipct_raw.notna().sum()),
        int((bmipct_raw.notna() & ~bmipct_raw.between(0, 100)).sum()),
        int(bmipct_raw.between(0, 100).sum()),
    ],
})

bmipct_summary = bmipct_raw.describe().to_frame("BMIPCT").reset_index()
bmipct_summary = bmipct_summary.rename(columns={"index": "statistic"})

bmipct_check.to_csv(TAB_DIR / "01_bmipct_data_check.csv", index=False)
bmipct_summary.to_csv(TAB_DIR / "01_bmipct_summary_statistics.csv", index=False)

display(bmipct_check)
display(bmipct_summary)

,metric,value
0,Total rows,14041
1,Missing values,979
2,Non-missing values,13062
3,Invalid non-missing values outside 0-100,0
4,Final valid response-variable rows,13062


,statistic,BMIPCT
0,count,13062.0000
1,mean,64.8207
2,std,27.5168
3,min,0.0000
4,25%,45.1663
5,50%,70.1385
6,75%,89.4510
7,max,99.9392


### 2.3 Final valid analysis rows


In [4]:
valid_rows = (
    raw["PerceptionOfWeight"].isin(valid_perception_codes)
    & raw["BMIPCT"].between(0, 100)
)

final_valid_check = pd.DataFrame({
    "metric": [
        "Total rows",
        "PerceptionOfWeight valid rows",
        "BMIPCT valid rows",
        "Final valid analysis rows",
        "Rows excluded from analysis",
    ],
    "value": [
        len(raw),
        int(raw["PerceptionOfWeight"].isin(valid_perception_codes).sum()),
        int(raw["BMIPCT"].between(0, 100).sum()),
        int(valid_rows.sum()),
        int(len(raw) - valid_rows.sum()),
    ],
})

final_valid_check.to_csv(TAB_DIR / "01_final_valid_analysis_rows.csv", index=False)
display(final_valid_check)


,metric,value
0,Total rows,14041
1,PerceptionOfWeight valid rows,13794
2,BMIPCT valid rows,13062
3,Final valid analysis rows,12853
4,Rows excluded from analysis,1188


## 3. Clean and Recode Analysis Data

Cleaning rules:

1. Select only `PerceptionOfWeight` and `BMIPCT`.
2. Drop rows with missing values in either variable.
3. Keep only valid `PerceptionOfWeight` values: 1, 2, 3, 4, and 5.
4. Keep only valid `BMIPCT` values from 0 to 100.
5. Recode `PerceptionOfWeight` into readable category labels.
6. Save the cleaned dataset to `data/processed/weight_perception_bmipct_cleaned.csv`.



In [5]:
PROCESSED_PATH = ROOT / "data" / "processed" / "weight_perception_bmipct_cleaned.csv"
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)

analysis = raw[["PerceptionOfWeight", "BMIPCT"]].copy()

rows_start = len(analysis)
rows_missing = int(analysis.isna().any(axis=1).sum())

analysis = analysis.dropna(subset=["PerceptionOfWeight", "BMIPCT"]).copy()

valid_codes = [1, 2, 3, 4, 5]
invalid_perception = int((~analysis["PerceptionOfWeight"].isin(valid_codes)).sum())
invalid_bmipct = int((~analysis["BMIPCT"].between(0, 100)).sum())

analysis = analysis[
    analysis["PerceptionOfWeight"].isin(valid_codes)
    & analysis["BMIPCT"].between(0, 100)
].copy()
analysis = analysis.reset_index(drop=True)

weight_perception_labels = {
    1: "Very underweight",
    2: "Slightly underweight",
    3: "About the right weight",
    4: "Slightly overweight",
    5: "Very overweight",
}
weight_perception_order = list(weight_perception_labels.values())

analysis["PerceptionOfWeight"] = analysis["PerceptionOfWeight"].astype(int)
analysis["WeightPerception"] = analysis["PerceptionOfWeight"].map(weight_perception_labels)
analysis["WeightPerception"] = pd.Categorical(
    analysis["WeightPerception"],
    categories=weight_perception_order,
    ordered=True
)

analysis.to_csv(PROCESSED_PATH, index=False)

cleaning_summary = pd.DataFrame({
    "metric": [
        "Rows at start",
        "Rows with missing analysis variable",
        "Invalid PerceptionOfWeight rows after dropping missing",
        "Invalid BMIPCT rows after dropping missing",
        "Final cleaned rows",
        "Rows removed from analysis",
    ],
    "value": [
        rows_start,
        rows_missing,
        invalid_perception,
        invalid_bmipct,
        len(analysis),
        rows_start - len(analysis),
    ],
})

cleaning_summary.to_csv(TAB_DIR / "01_cleaning_summary.csv", index=False)
print("Rows saved for cleaned analysis data:", len(analysis))
print("Saved cleaned dataset to:", PROCESSED_PATH)
print("Saved cleaning summary table to:", TAB_DIR / "01_cleaning_summary.csv")

Rows saved for cleaned analysis data: 12853
Saved cleaned dataset to: C:\Users\asd45\OneDrive\Desktop\cycle4\data\processed\weight_perception_bmipct_cleaned.csv
Saved cleaning summary table to: C:\Users\asd45\OneDrive\Desktop\cycle4\outputs\tables\01_cleaning_summary.csv
